In [1]:
import pandas as pd

pairs = pd.read_excel("conjoint_pairs_150.xlsx")

In [2]:
attrs = {
    "country": "Страна происхождения",
    "motivation": "Мотивация приезда",
    "employer": "Тип работодателя",
    "gender": "Пол",
    "age": "Возраст",
    "occupation": "Сфера занятости",
    "language": "Владение русским языком",
}

intro = "Ниже вы увидите двух кандидатов."
question = "Если бы Вам нужно было выбрать между ними, кому из этих двух мигрантов следует отдать приоритет для предоставления вида на жительство?"

In [3]:
def make_rows(row, side):
    result = ""
    for attr, label in attrs.items():
        value = row[f"{side}_{attr}"]
        result += f'<tr><td style="padding-bottom: 12px;"><strong>{label}:</strong> {value}</td></tr>'
    return result

In [4]:
def make_block(row):
    a_rows = make_rows(row, "A")
    b_rows = make_rows(row, "B")
    return f"""<p>{intro}</p>
<table style="width: 100%;">
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 20px;">
      <p><strong>Кандидат А</strong></p>
      <table style="width: 100%;">{a_rows}</table>
    </td>
    <td style="width: 50%; vertical-align: top; padding-left: 20px;">
      <p><strong>Кандидат Б</strong></p>
      <table style="width: 100%;">{b_rows}</table>
    </td>
  </tr>
</table>
<p style="margin-top: 20px;"><strong>{question}</strong></p>"""


blocks = []
for i in range(len(pairs)):
    blocks.append(make_block(pairs.iloc[i]))


In [6]:
letters = {"country": "C", "motivation": "M", "employer": "E",
           "gender": "G", "age": "A", "occupation": "O", "language": "L"}

codes = {}
for attr in attrs:
    unique = sorted(set(pairs[f"A_{attr}"].astype(str)) | set(pairs[f"B_{attr}"].astype(str)))
    codes[attr] = {v: i + 1 for i, v in enumerate(unique)}

def make_label(row):
    a = "".join(f"{letters[attr]}{codes[attr][str(row[f'A_{attr}'])]}" for attr in attrs)
    b = "".join(f"{letters[attr]}{codes[attr][str(row[f'B_{attr}'])]}" for attr in attrs)
    return f"{a} / {b}"

variable_names = [f"CJ{i + 1}" for i in range(len(pairs))]
labels = [make_label(pairs.iloc[i]) for i in range(len(pairs))]

print("Расшифровка кодов:")
for attr, mapping in codes.items():
    print(f"  {letters[attr]} = {attr}")
    for value, code in mapping.items():
        print(f"    {letters[attr]}{code} = {value}")

Расшифровка кодов:
  C = country
    C1 = Беларусь
    C2 = Венгрия
    C3 = Индия
    C4 = Пакистан
    C5 = Румыния
    C6 = Узбекистан
    C7 = Украина
  M = motivation
    M1 = Воссоединение с супругом(ой)
    M2 = Поиск работы
    M3 = Политическая нестабильность
    M4 = Учёба
  E = employer
    E1 = Государственная организация
    E2 = Небольшая частная компания
  G = gender
    G1 = Женщина
    G2 = Мужчина
  A = age
    A1 = 21
    A2 = 48
    A3 = 62
  O = occupation
    O1 = Врач
    O2 = Программист
    O3 = Строитель
    O4 = Сфера услуг (общепит)
  L = language
    L1 = Говорит плохо
    L2 = Говорит свободно


In [9]:
with open("conjoint_pairsf.txt", "w", encoding="utf-8") as f:
    for i, block in enumerate(blocks):
        f.write(f"Пара {i + 1}\n")
        f.write(f"Variable name: {variable_names[i]}\n")
        f.write(f"Label: {labels[i]}\n")
        f.write(block)
        f.write("\n\n")

# html для просмотра в браузере
with open("conjoint_pairs.html", "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'></head><body>")
    for i, block in enumerate(blocks):
        f.write(f"<h3>Пара {i + 1}</h3>")
        f.write(block)
        f.write("<hr>")
    f.write("</body></html>")

In [10]:
mapping = pairs.copy()
mapping.insert(0, "variable_name", variable_names)
mapping.insert(1, "label", labels)
mapping.to_csv("cj_mapping.csv", index=False, encoding="utf-8-sig")
mapping.head()

,variable_name,label,pair_id,A_country,B_country,A_motivation,B_motivation,A_employer,B_employer,A_gender,B_gender,A_age,B_age,A_occupation,B_occupation,A_language,B_language
0,CJ1,C2M1E1G1A3O1L1 / C1M3E1G1A3O2L1,1,Венгрия,Беларусь,Воссоединение с супругом(ой),Политическая нестабильность,Государственная организация,Государственная организация,Женщина,Женщина,62,62,Врач,Программист,Говорит плохо,Говорит плохо
1,CJ2,C4M2E1G2A1O2L2 / C4M3E1G1A1O2L1,2,Пакистан,Пакистан,Поиск работы,Политическая нестабильность,Государственная организация,Государственная организация,Мужчина,Женщина,21,21,Программист,Программист,Говорит свободно,Говорит плохо
2,CJ3,C6M1E1G2A1O4L2 / C7M1E2G2A1O4L2,3,Узбекистан,Украина,Воссоединение с супругом(ой),Воссоединение с супругом(ой),Государственная организация,Небольшая частная компания,Мужчина,Мужчина,21,21,Сфера услуг (общепит),Сфера услуг (общепит),Говорит свободно,Говорит свободно
3,CJ4,C6M2E1G2A2O1L1 / C6M2E1G1A2O4L1,4,Узбекистан,Узбекистан,Поиск работы,Поиск работы,Государственная организация,Государственная организация,Мужчина,Женщина,48,48,Врач,Сфера услуг (общепит),Говорит плохо,Говорит плохо
4,CJ5,C4M4E1G1A1O3L1 / C1M4E2G1A1O4L2,5,Пакистан,Беларусь,Учёба,Учёба,Государственная организация,Небольшая частная компания,Женщина,Женщина,21,21,Строитель,Сфера услуг (общепит),Говорит плохо,Говорит свободно
